# Load Libraries

In [13]:
import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from collections import Counter
from nltk.util import ngrams

In [14]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# Load Data

In [15]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# 1. Filtering Data

In [16]:
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [17]:
print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nClass Distribution:")
print(df['label'].value_counts())

Dataset Shape: (5572, 2)

Missing Values:
label      0
message    0
dtype: int64

Class Distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


## 2. Text Preprocessing

Text preprocessing is an important step in NLP because raw text may contain unnecessary characters and words that can affect the analysis.

In this project, I applied the following preprocessing steps:

1. **Lowercasing:** Convert all text into lowercase to make words consistent.
2. **Removing Numbers:** Remove numerical values because they are not needed for our text analysis.
3. **Removing Punctuation:** Remove punctuation marks such as commas, periods, and question marks.
4. **Removing Special Characters:** Remove special characters such as currency symbols and other non-alphabetic characters.
5. **Tokenization:** Split each message into individual words using NLTK's `word_tokenize()`.
6. **Stopwords Removal:** Remove common words that do not add significant meaning to the analysis.
7. **Lemmatization:** Convert words into their base form using WordNetLemmatizer.

After preprocessing, each message is represented as a list of clean tokens that can be used in the N-Gram Language Model.

In [18]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens

In [19]:
df['processed_message'] = df['message'].apply(preprocess_text)
df.head()

,label,message,processed_message
0,ham,"Go until jurong point, crazy.. Available only ...","[go, jurong, point, crazy, available, bugis, n..."
1,ham,Ok lar... Joking wif u oni...,"[ok, lar, joking, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, wkly, comp, win, fa, cup, final,..."
3,ham,U dun say so early hor... U c already then say...,"[u, dun, say, early, hor, u, c, already, say]"
4,ham,"Nah I don't think he goes to usf, he lives aro...","[nah, dont, think, go, usf, life, around, though]"


In [20]:
print("Original Message:")
print(df['message'].iloc[0])

print("\nProcessed Message:")
print(df['processed_message'].iloc[0])

Original Message:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Processed Message:
['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']


## 3. N-Gram Language Model

An N-Gram Language Model is a statistical language model that predicts the probability of a word based on the previous words.

In this project, I used a **Bigram Model (N=2)**, which considers two consecutive words at a time.

In [21]:
all_tokens = []

for tokens in df['processed_message']:
    all_tokens.extend(tokens)

print("Total number of tokens:", len(all_tokens))
print("First 20 tokens:")
print(all_tokens[:20])

Total number of tokens: 49528
First 20 tokens:
['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat', 'ok', 'lar', 'joking', 'wif']


### 3.1 Unigram Model

A Unigram represents a single word.

The frequency of each word is calculated using the Counter function.

In [22]:
unigram_counts = Counter(all_tokens)
print("Most common 10 words:")
print(unigram_counts.most_common(10))

Most common 10 words:
[('u', 1206), ('call', 603), ('im', 474), ('get', 401), ('ur', 384), ('go', 308), ('dont', 290), ('free', 278), ('ok', 277), ('ltgt', 276)]


### 3.2 Bigram Model

A Bigram represents a pair of consecutive words.

For example, the sentence:

`I love machine learning`

Produces the following bigrams:

- (I, love)
- (love, machine)
- (machine, learning)

In [23]:
all_bigrams = []

for tokens in df['processed_message']:
    message_bigrams = list(ngrams(tokens, 2))
    all_bigrams.extend(message_bigrams)

bigram_counts = Counter(all_bigrams)

print("Most common 10 bigrams:")
print(bigram_counts.most_common(10))

Most common 10 bigrams:
[(('gon', 'na'), 58), (('please', 'call'), 55), (('call', 'later'), 52), (('ill', 'call'), 48), (('u', 'r'), 44), (('let', 'know'), 41), (('dont', 'know'), 39), (('sorry', 'ill'), 39), (('r', 'u'), 39), (('wan', 'na'), 38)]


### 3.3 Conditional Probability

The probability of a word occurring after another word is calculated using:

\[
P(w_2|w_1)=\frac{Count(w_1,w_2)}{Count(w_1)}
\]

Where:

- Count(w1, w2): Number of occurrences of the bigram.
- Count(w1): Number of occurrences of the first word as a context.

This probability helps us understand how likely a word is to follow another word.

In [24]:
context_counts = Counter()

for tokens in df['processed_message']:
    context_counts.update(tokens[:-1])

# Calculate probability for all bigrams
bigram_probabilities = {}

for bigram, count in bigram_counts.items():

    word1, word2 = bigram

    denominator = context_counts[word1]

    probability = count / denominator

    bigram_probabilities[bigram] = probability

print("Most Common 10 Bigrams and Their Probabilities:\n")

for bigram, count in bigram_counts.most_common(10):

    probability = bigram_probabilities[bigram]

    print(f"Bigram: {bigram}")
    print(f"Count: {count}")
    print(f"Probability: {probability:.4f}")
    print(f"Percentage: {probability * 100:.2f}%")
    print("-" * 40)

Most Common 10 Bigrams and Their Probabilities:

Bigram: ('gon', 'na')
Count: 58
Probability: 0.9831
Percentage: 98.31%
----------------------------------------
Bigram: ('please', 'call')
Count: 55
Probability: 0.4661
Percentage: 46.61%
----------------------------------------
Bigram: ('call', 'later')
Count: 52
Probability: 0.1002
Percentage: 10.02%
----------------------------------------
Bigram: ('ill', 'call')
Count: 48
Probability: 0.2017
Percentage: 20.17%
----------------------------------------
Bigram: ('u', 'r')
Count: 44
Probability: 0.0390
Percentage: 3.90%
----------------------------------------
Bigram: ('let', 'know')
Count: 41
Probability: 0.4457
Percentage: 44.57%
----------------------------------------
Bigram: ('dont', 'know')
Count: 39
Probability: 0.1349
Percentage: 13.49%
----------------------------------------
Bigram: ('sorry', 'ill')
Count: 39
Probability: 0.2600
Percentage: 26.00%
----------------------------------------
Bigram: ('r', 'u')
Count: 39
Probability

### 3.5 Calculating Unigram Probabilities

Before calculating sentence probabilities, we first calculate the probability of each individual word.

The Unigram Probability is calculated by dividing the word frequency by the total number of tokens in the dataset.

\[
P(w)=\frac{Count(w)}{Total\ Tokens}
\]

In [25]:
total_tokens = len(all_tokens)

unigram_probabilities = {}

for word, count in unigram_counts.items():

    probability = count / total_tokens

    unigram_probabilities[word] = probability


print("Total Tokens:", total_tokens)

print("\nSample Unigram Probabilities:")

for word in list(unigram_probabilities)[:10]:

    print(f"{word}: {unigram_probabilities[word]:.6f}")

Total Tokens: 49528

Sample Unigram Probabilities:
go: 0.006219
jurong: 0.000020
point: 0.000666
crazy: 0.000303
available: 0.000323
bugis: 0.000141
n: 0.002907
great: 0.002241
world: 0.000727
la: 0.000182


### 3.6 Selecting Three Messages

In this step, I select three messages from the original dataset to calculate their probabilities.

The messages are selected from the preprocessed dataset to ensure that the calculations use the same tokens used to build the Bigram Model.

In [26]:
# Select messages that contain at least two words

sentence_candidates = []

for index, tokens in enumerate(df['processed_message']):

    if len(tokens) >= 2:

        sentence_candidates.append((index, tokens))


# Calculate the percentage of known bigrams in each message

sentence_scores = []

for index, tokens in sentence_candidates:

    message_bigrams = list(ngrams(tokens, 2))

    known_bigrams = sum(
        1 for bigram in message_bigrams
        if bigram in bigram_counts
    )

    coverage = known_bigrams / len(message_bigrams)

    sentence_scores.append((index, tokens, coverage))


# Select three messages with the highest bigram coverage

sentence_scores = sorted(
    sentence_scores,
    key=lambda x: x[2],
    reverse=True
)

selected_sentences = sentence_scores[:3]


print("Selected Messages:\n")

for i, (index, tokens, coverage) in enumerate(selected_sentences, 1):

    print(f"Sentence {i}:")
    print("Original:", df['message'].iloc[index])
    print("Processed:", tokens)
    print("Bigram Coverage:", f"{coverage * 100:.2f}%")
    print("-" * 50)

Selected Messages:

Sentence 1:
Original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Processed: ['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']
Bigram Coverage: 100.00%
--------------------------------------------------
Sentence 2:
Original: Ok lar... Joking wif u oni...
Processed: ['ok', 'lar', 'joking', 'wif', 'u', 'oni']
Bigram Coverage: 100.00%
--------------------------------------------------
Sentence 3:
Original: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
Processed: ['free', 'entry', 'wkly', 'comp', 'win', 'fa', 'cup', 'final', 'tkts', 'st', 'may', 'text', 'fa', 'receive', 'entry', 'questionstd', 'txt', 'ratetcs', 'apply', 'over']
Bigram Coverage: 100.00%
--------------------------------------------------


### 3.7 Calculating Sentence Probabilities

Now, we calculate the probability of three selected messages using the Bigram Language Model.

Based on the Markov Assumption, the sentence probability is calculated by multiplying:

1. The Unigram probability of the first word.
2. The conditional probability of every following word given the previous word.

The formula is:

\[
P(S)=P(w_1)\times P(w_2|w_1)\times
P(w_3|w_2)\times ... \times P(w_n|w_{n-1})
\]

The calculation steps and the probability of each Bigram will be displayed clearly for every selected message.

In [27]:
import math

def calculate_sentence_probability(tokens):

    # Calculate the probability of the first word

    first_word = tokens[0]

    first_probability = unigram_probabilities.get(first_word, 0)

    sentence_probability = first_probability

    print(f"P({first_word}) = {first_probability:.8f}")

    # Calculate the probability of each Bigram

    for i in range(1, len(tokens)):

        word1 = tokens[i - 1]
        word2 = tokens[i]

        bigram = (word1, word2)

        bigram_probability = bigram_probabilities.get(bigram, 0)

        print(f"P({word2} | {word1}) = {bigram_probability:.8f}")

        sentence_probability *= bigram_probability

    return sentence_probability


# Calculate probabilities for the three selected messages

for i, (index, tokens, coverage) in enumerate(selected_sentences, 1):

    print(f"\nSentence {i}")
    print("=" * 60)

    print("Original Message:")
    print(df['message'].iloc[index])

    print("\nProcessed Tokens:")
    print(tokens)

    print("\nProbability Calculation:")

    probability = calculate_sentence_probability(tokens)

    print("\nFinal Sentence Probability:")
    print(f"{probability:.12e}")

    print("-" * 60)


Sentence 1
Original Message:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Processed Tokens:
['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']

Probability Calculation:
P(go) = 0.00621870
P(jurong | go) = 0.00333333
P(point | jurong) = 1.00000000
P(crazy | point) = 0.03448276
P(available | crazy) = 0.07142857
P(bugis | available) = 0.07142857
P(n | bugis) = 0.20000000
P(great | n) = 0.00709220
P(world | great) = 0.00934579
P(la | world) = 0.02941176
P(e | la) = 0.14285714
P(buffet | e) = 0.01204819
P(cine | buffet) = 0.50000000
P(got | cine) = 0.14285714
P(amore | got) = 0.00411523
P(wat | amore) = 1.00000000

Final Sentence Probability:
7.193866069102e-22
------------------------------------------------------------

Sentence 2
Original Message:
Ok lar... Joking wif u oni...

Processed Tokens:
['ok', 'lar', 'joking', 'wif', 'u', 'oni']

P